Wrangling Data

In [7]:
import pandas as pd

sellers = pd.read_csv('Sellers.csv')
products = pd.read_csv('Products.csv')
suppliers = pd.read_csv('Suppliers.csv')
transactions = pd.read_csv('Transactions.csv')
logs = pd.read_csv('Logs.csv')

In [8]:
files = [
    "Logs.csv",
    "Transactions.csv",
    "Suppliers.csv",
    "Products.csv",
    "Sellers.csv"
]

# Loop untuk membaca dan menampilkan 5 baris pertama setiap file
for file in files:
    df = pd.read_csv(file)
    print(f"Menampilkan: {file}")
    display(df.head())
    print("\n")

Menampilkan: Logs.csv


,id_log,id_produk,id_seller,tanggal_log,jenis_log,stok_sebelum,stok_sesudah,jumlah_perubahan,keterangan
0,L0304,P008,U002,2024-01-01,penyesuaian_stok,35,34,-1,Penyesuaian stok hasil audit
1,L0095,P005,U001,2024-01-02,penyesuaian_stok,53,63,10,Penyesuaian stok hasil audit
2,L0099,P017,U004,2024-01-06,stok_menipis,14,10,-4,Stok mendekati batas minimum
3,L0114,P008,U002,2024-01-09,penyesuaian_stok,20,12,-8,Penyesuaian stok hasil audit
4,L0324,P004,U001,2024-01-11,restock,34,60,26,Penambahan stok dari supplier




Menampilkan: Transactions.csv


,id_transaksi,id_seller,id_produk,tanggal,jenis_transaksi,kategori,qty,harga_satuan,total_harga,metode_pembayaran,event,diskon,status_transaksi
0,T00001,U006,P026,2026-05-28,pemasukan,penjualan_produk,1,85000,85000,QRIS,NaN,NaN,sukses
1,T00002,U006,P030,2026-05-15,pemasukan,penjualan_produk,1,55000,55000,Cash,NaN,NaN,sukses
2,T00003,U005,P025,2026-03-28,pemasukan,jasa_laundry,5,12000,60000,QRIS,NaN,NaN,sukses
3,T00004,U005,P021,2026-08-04,pemasukan,jasa_laundry,2,12000,24000,Transfer,NaN,NaN,sukses
4,T00005,U003,P012,2025-07-10,pengeluaran,restock_barang,3,18000,54000,Cash,NaN,NaN,sukses




Menampilkan: Suppliers.csv


,id_supplier,nama_supplier,kategori_supplier,lokasi
0,S001,PT Kopi Nusantara,Biji Kopi & Minuman,Bandung
1,S002,CV Sembako Jaya,Sembako & FMCG,Bekasi
2,S003,UD Dapur Nusantara,Bahan Makanan,Yogyakarta
3,S004,UD Bakery Fresh,Bakery & Snack,Jakarta
4,S005,CV Kebutuhan Rumah,Household & Laundry,Tangerang




Menampilkan: Products.csv


,id_produk,id_seller,id_supplier,nama_produk,kategori_produk,harga_jual,harga_modal,stok_awal,minimum_stok,status_produk,tanggal_input_produk
0,P001,U001,S001,Kopi Susu,Minuman,18000,9000,100,20,aktif,2024-01-15
1,P002,U001,S001,Americano,Minuman,15000,7000,100,20,aktif,2024-01-15
2,P003,U001,S001,Latte,Minuman,20000,10000,90,18,aktif,2024-01-15
3,P004,U001,S004,Croissant,Makanan,18000,9000,60,12,aktif,2024-01-16
4,P005,U001,S004,Cookies,Snack,12000,6000,70,15,aktif,2024-01-16




Menampilkan: Sellers.csv


,id_seller,nama_usaha,jenis_usaha,lokasi_usaha,tanggal_bergabung
0,U001,Warkop GenZ,Kafe,Bandung,2024-01-10
1,U002,Warung Madura,Retail,Bekasi,2024-01-15
2,U003,Makan Yuk,Kuliner,Yogyakarta,2025-02-01
3,U004,Sweet Bite Bakery,Bakery,Jakarta,2025-03-01
4,U005,LaundryKita,Laundry,Tangerang,2026-03-15


## 1. Analisis Missing Values (Nilai Kosong)

Logs, Suppliers, Products, Sellers: Tidak ditemukan missing value (0%). Semua data terisi dengan lengkap.

event: 100% kosong (5 dari 5 baris). Ini wajar jika transaksi dilakukan di hari biasa tanpa kampanye promo tertentu.

diskon: 100% kosong. Menunjukkan tidak ada potongan harga pada transaksi sampel ini.

status_transaksi: Ada kejanggalan pada tampilan teks awal Anda (terlihat NaN di kolom ke-12), namun pada kolom terakhir tertulis "sukses". Jika mengikuti struktur tabel, pastikan tidak ada pergeseran kolom saat proses import.

## 2. Temuan Penting & Anomali Data
Selain nilai kosong, saya menemukan beberapa poin menarik yang bisa memengaruhi integritas data Anda:

Inkoneksitas Seller:
Di Transactions.csv, terdapat id_seller: U006. Namun, jika kita melihat ke Sellers.csv, ID seller hanya sampai U005. Ini disebut orphan record (data yatim), di mana ada transaksi dari penjual yang datanya tidak terdaftar di tabel master Seller.

Inkoneksitas Produk:
Hal serupa terjadi pada id_produk: P026, P030, P025, P021. Produk-produk ini muncul di transaksi tetapi tidak ada di master Products.csv (yang hanya menampilkan P001-P005).

Rentang Waktu (Timeline):
Data memiliki rentang waktu yang sangat jauh:

Logs & Products mencatat aktivitas di awal 2024.
Transactions mencatat aktivitas hingga Mei & Agustus 2026.

Potensi Inkonsistensi Stok:
Pada Logs.csv baris 0 dan 3, produk P008 mengalami pengurangan stok lewat "Penyesuaian stok hasil audit". Karena alasannya adalah audit dan jumlahnya cukup sering (total berkurang 9 unit), ini bisa menjadi indikasi adanya kebocoran stok, kehilangan barang, atau kerusakan yang tidak tercatat di transaksi reguler.

## 3. Rekomendasi Tindakan
Sinkronisasi Master Data: Segera update Sellers.csv dan Products.csv agar mencakup ID yang muncul di transaksi (U006, P021, dll) untuk menghindari kesalahan saat penarikan laporan.

Validasi Kolom Status: Periksa kembali pemisahan kolom pada file CSV transaksi Anda, pastikan antara kolom event, diskon, dan status_transaksi tidak tertukar atau bergeser.

Audit Stok P008: Lakukan pengecekan fisik lebih ketat untuk seller U002 karena frekuensi penyesuaian stok negatifnya cukup mencolok.

In [9]:
# Konversi tanggal ke format datetime
transactions['tanggal'] = pd.to_datetime(transactions['tanggal'])
logs['tanggal_log'] = pd.to_datetime(logs['tanggal_log'])
sellers['tanggal_bergabung'] = pd.to_datetime(sellers['tanggal_bergabung'])

# Pembersihan: Filter hanya transaksi yang sukses
transactions_clean = transactions[transactions['status_transaksi'] == 'sukses'].copy()

# Penggabungan Data Master (Produk + Seller + Supplier)
product_master = products.merge(sellers, on='id_seller', how='left') \
                         .merge(suppliers, on='id_supplier', how='left', suffixes=('_usaha', '_supplier'))

In [10]:
# Membuat Dataset Utama Transaksi
df_final = transactions_clean.merge(product_master, on=['id_produk', 'id_seller'], how='left')

# Rekayasa Fitur Menghitung Profit
df_final['total_modal'] = df_final['harga_modal'] * df_final['qty']
df_final['profit'] = df_final['total_harga'] - df_final['total_modal']

# Menampilkan hasil wrangling
print("Informasi Dataset Akhir:")
print(df_final.info())
print("\nLima baris pertama dataset transaksi lengkap:")
print(df_final[['id_transaksi', 'nama_usaha', 'nama_produk', 'total_harga', 'profit']].head())

Informasi Dataset Akhir:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1929 entries, 0 to 1928
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_transaksi          1929 non-null   object        
 1   id_seller             1929 non-null   object        
 2   id_produk             1929 non-null   object        
 3   tanggal               1929 non-null   datetime64[ns]
 4   jenis_transaksi       1929 non-null   object        
 5   kategori              1929 non-null   object        
 6   qty                   1929 non-null   int64         
 7   harga_satuan          1929 non-null   int64         
 8   total_harga           1929 non-null   int64         
 9   metode_pembayaran     1929 non-null   object        
 10  event                 180 non-null    object        
 11  diskon                68 non-null     float64       
 12  status_transaksi      1929 non-null   object       